In [18]:
from dotenv import load_dotenv
import os

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore

In [20]:
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [43]:
import os
from langchain_community.document_loaders import PyPDFLoader

directory_path = r"D:\\03_Study\\GenAI\\3.3 Traditional RAG 02\\Source Files\\"

all_documents = []

for filename in os.listdir(directory_path):
    print(filename)
    if filename.lower().endswith(".pdf"):
        file_path = os.path.join(directory_path, filename)
        loader = PyPDFLoader(file_path)
        documents = loader.load()  # returns a list of Document objects (one per page)
        all_documents.extend(documents)

print(f"Loaded {len(all_documents)} pages total from directory.")

Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 31 0 (offset 0)
Ignoring wrong pointing object 84 0 (offset 0)


Incremental+Data+Ingestion+from+Files.pdf
Lakeflow+Declarative+Pipelines (1).pdf
Lakeflow+Declarative+Pipelines.pdf
Structured+Streaming.pdf
Loaded 45 pages total from directory.


In [44]:
print(all_documents)

[Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227195542Z00'00'", 'moddate': "D:20231227195542Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Incremental+Data+Ingestion+from+Files.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='Incremental Data \nIngestion from Files'), Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227195542Z00'00'", 'moddate': "D:20231227195542Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Incremental+Data+Ingestion+from+Files.pdf', 'total_pages': 11, 'page': 1, 'page_label': '2'}, page_content='Learning Objectives\nuWhat is incremental data Ingestion from file\nuCOPY INTO\nuAuto Loader\nDerar Alhussein © Udemy | Databricks Certified Data Engineer Associate -Preparation'), Document(metadata={'produ

In [47]:
pages = list(all_documents)
print(f"Loaded {len(pages)} pages total from all PDF files.")

Loaded 45 pages total from all PDF files.


In [48]:
print(pages)

[Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227195542Z00'00'", 'moddate': "D:20231227195542Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Incremental+Data+Ingestion+from+Files.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='Incremental Data \nIngestion from Files'), Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227195542Z00'00'", 'moddate': "D:20231227195542Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Incremental+Data+Ingestion+from+Files.pdf', 'total_pages': 11, 'page': 1, 'page_label': '2'}, page_content='Learning Objectives\nuWhat is incremental data Ingestion from file\nuCOPY INTO\nuAuto Loader\nDerar Alhussein © Udemy | Databricks Certified Data Engineer Associate -Preparation'), Document(metadata={'produ

In [23]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,#hyperparameter
    chunk_overlap=50 #hyperparemeter
)

In [49]:
split_docs = splitter.split_documents(pages)

In [45]:
print(pages)

[Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227202709Z00'00'", 'moddate': "D:20231227202709Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Structured+Streaming.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Structured Streaming'), Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227202709Z00'00'", 'moddate': "D:20231227202709Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Structured+Streaming.pdf', 'total_pages': 12, 'page': 1, 'page_label': '2'}, page_content='Learning Objectives\nuProcess streaming data\nuDataStreamReader\nuDataStreamWriter\nDerar Alhussein © Udemy | Databricks Certified Data Engineer Associate -Preparation'), Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'c

In [25]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [26]:
pc = Pinecone()

In [27]:
index_name = "traditional-rag-03"

In [28]:
#Creating the index
if not pc.has_index(index_name):
    pc.create_index(
    name = index_name,
    dimension = 3072,
    metric = "cosine",
    spec = ServerlessSpec(cloud = "aws", region = "us-east-1")
    )


In [29]:
#Loading the Index
index = pc.Index(index_name)

In [30]:
vector_store = PineconeVectorStore(index = index, embedding = embeddings)

In [31]:
split_docs

[Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227202709Z00'00'", 'moddate': "D:20231227202709Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Structured+Streaming.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Structured Streaming'),
 Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227202709Z00'00'", 'moddate': "D:20231227202709Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Structured+Streaming.pdf', 'total_pages': 12, 'page': 1, 'page_label': '2'}, page_content='Learning Objectives\nuProcess streaming data\nuDataStreamReader\nuDataStreamWriter\nDerar Alhussein © Udemy | Databricks Certified Data Engineer Associate -Preparation'),
 Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 

In [50]:
vector_store.add_documents(documents = split_docs)

['e8a77784-36e7-4db2-b103-52fe55403da2',
 '2f0214bb-62b1-4202-9595-b78cc5e27a94',
 '3617fbfe-d851-490c-af69-1efa305ca1c1',
 'b39ab2fc-3188-479b-807d-7cac76f7ae11',
 '6ebd4c08-3d6d-418f-9fc3-50b6a8089254',
 '9da43135-9ad5-4239-bcec-1402a7154e23',
 'fc21cd1f-68e1-4297-a609-1b37b7d29de9',
 '67fd012e-b089-47de-b87d-d7659ee5d457',
 '3cad78c8-3fff-41ab-aa3f-f1480bc48b07',
 '2000920a-5661-417f-9603-05e940a8ece6',
 '1026e441-db8a-4c2e-bfd0-13757889ec67',
 '30dd20d1-4d0b-45e9-aac5-d29061bc8df3',
 '3f48434a-af15-4977-8326-9a74a40dc593',
 '7a8b5784-8b58-4c1e-bb3b-0c139ca95725',
 '5059bc48-d536-4d4d-b749-fd6ecff23feb',
 'c318cfcb-d844-4876-aab6-7019cdac157e',
 '400508f7-af2b-4fbc-adce-78727c8d504d',
 'f82df34b-87af-4216-bad9-9acd58ad5956',
 '49190eaa-0c00-47a9-ab16-a25979ec8d15',
 '07b65139-13b2-4c50-98f9-e4aa6e0a634b',
 '4a9ceaf0-14aa-4b3c-be20-c40fff5eb5c3',
 '17257e49-fd15-4b51-bf9f-e95cbd8606f9',
 '6ac585a0-c513-4998-9929-ad62c07a3003',
 '23a418e5-9efa-4168-921d-99e42788e4fa',
 '0657c5b7-3789-

In [33]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [34]:
prompt=PromptTemplate(
    template="""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:""",
    input_variables=['context', 'question']
)

In [35]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [36]:
retriever=vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.7} #hyperparameter
)

In [37]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model='gemini-3.6-flash')

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [38]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [41]:
split_docs

[Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227202709Z00'00'", 'moddate': "D:20231227202709Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Structured+Streaming.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'text': 'Structured Streaming'}, page_content='Structured Streaming'),
 Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20231227202709Z00'00'", 'moddate': "D:20231227202709Z00'00'", 'source': 'D:\\\\03_Study\\\\GenAI\\\\3.3 Traditional RAG 02\\\\Source Files\\\\Structured+Streaming.pdf', 'total_pages': 12, 'page': 1, 'page_label': '2', 'text': 'Learning Objectives\nuProcess streaming data\nuDataStreamReader\nuDataStreamWriter\nDerar Alhussein © Udemy | Databricks Certified Data Engineer Associate -Preparation'}, page_content='Learning Objectives\nuProcess streaming data\nu

In [53]:
rag_chain.invoke("What is a LDP?")

'Based on the provided context, LDP refers to a pipeline-based framework using the `pyspark.pipelines` module to define data processing objects. It allows users to create tables, materialized views, and temporary views using decorators like `@dp.table`. Compared to standard Spark code, LDP simplifies defining streaming data pipelines.'